### Processing EMTP-RV Parametric Studio outputs

In [ ]:
import pickle
import numpy as np

In [ ]:
from simulation import Simulations

In [ ]:
from utils import create_white_list
from readme import read_me

In [ ]:
# ========================
# *** INPUT PARAMETERS ***
# ========================
# Directory path (EDIT):
simulation_path = ''

# Model file name without extension (EDIT):
name = 'IEEE39_Wind_v5_x'

# Power system variant.
variant = 'V0'

# Short-circuit duration (EDIT):
sc_time = 'T100ms'  # 'T100ms' or 'T300ms'

In [ ]:
# List of machines that are excluded.
if variant == 'V0':
    exclude = []
elif variant == 'V1':
    exclude = [5, 8]
elif variant in ['V2', 'V3']:
    exclude = [3, 5, 8, 9]
elif variant == 'V4':
    exclude = [3, 5, 8, 9, 10]
else:
    raise ValueError()

# List of machine variable names:
varnames = [
    '/Teta_1_SM1',   # rotor angle
    '/Omega_1_SM1',  # rotor speed
    '/PowerAng_SM1', # power angle
    '/Pe_SM1',       # electrical power
    '/vd_SM1',  # d-axis stator voltage
    '/id_SM1',  # d-axis stator current
    '/Ef_SM1',  # EMF voltage (q-axis)
    '/vq_SM1',  # q-axis stator voltage
    '/iq_SM1',  # q-axis stator current
]

# List of bus index values.
buses = np.arange(start=1, stop=30, dtype=int).tolist()
# Buses 30 to 38 are skipped, since they are
# between the generator and its step-up transformer.
buses.append(39)  # external network node
# New artificial nodes in the middle of each transmission line,
# except the line between bus 5 and 6 (which is not divided).
# There are no voltage measurements at these nodes.
middle_points = np.arange(start=40, stop=73, dtype=int).tolist()
# List of all nodes where short-circuits will be applied.
# A distinction is made between nodes (where SC is applied)
# and buses (where voltage is measured).
nodes = buses + middle_points

In [ ]:
# Generate the "white_list" variable.
white_list = create_white_list(varnames, exclude, buses, variant)
white_list

In [ ]:
# Build all simulations.
sims = Simulations(simulation_path, name, white_list)
sims.build_all_simulations()

In [ ]:
nsim = sims.get_nb_simu_tot()
print(f'Total no. of simulations: {nsim}')

# Dictionary keys for simulations which identify 
# SC type and node number of the fault location.
sim_keys = ['SC3-' + 'BUS'+str(k) for k in nodes]
sim_keys.extend(['SC2-' + 'BUS'+str(k) for k in nodes])
sim_keys.extend(['SC1-' + 'BUS'+str(k) for k in nodes])

if nsim != len(sim_keys):
    raise ValueError()

# Name pairs for renaming select columns.
name_pairs = {
    # Wind farm (WF) signals.
    'DEV2/P': 'WF/P',
    'DEV2/Q': 'WF/Q',
    'DEV2/V0': 'WF/V0',
    'DEV2/V1': 'WF/V1',
    'DEV2/V2': 'WF/V2',
    'DEV2/I0': 'WF/I0',
    'DEV2/I1': 'WF/I1',
    'DEV2/I2': 'WF/I2',
    'FFC_WP1/Wind_Turbine/PMSG_T_rotor': 'WF/PMSG_T_rotor',
    'FFC_WP1/Wind_Turbine/PMSG_w_rotor': 'WF/PMSG_w_rotor',
    'FFC_WP1/Converter_control/Control/Grid_Ctrl/FRT_flag': 'WF/FRT_flag',
    # PV plant (PV) signals.
    'DEV3/P': 'PV/P',
    'DEV3/Q': 'PV/Q',
    'DEV3/V0': 'PV/V0',
    'DEV3/V1': 'PV/V1',
    'DEV3/V2': 'PV/V2',
    'DEV3/I0': 'PV/I0',
    'DEV3/I1': 'PV/I1',
    'DEV3/I2': 'PV/I2',
    'WECC_PVPark_1/Converter_control/Control/GridControl_DLL/FRT_flag': 'PV/FRT_flag',
}
name_pairs_two = {
    # Wind farm 2 (WF2) signals.
    'DEV4/P': 'WF2/P',
    'DEV4/Q': 'WF2/Q',
    'DEV4/V0': 'WF2/V0',
    'DEV4/V1': 'WF2/V1',
    'DEV4/V2': 'WF2/V2',
    'DEV4/I0': 'WF2/I0',
    'DEV4/I1': 'WF2/I1',
    'DEV4/I2': 'WF2/I2',
    'FFC_WP2/Wind_Turbine/PMSG_T_rotor': 'WF2/PMSG_T_rotor',
    'FFC_WP2/Wind_Turbine/PMSG_w_rotor': 'WF2/PMSG_w_rotor',
    'FFC_WP2/Converter_control/Control/Grid_Ctrl/FRT_flag': 'WF2/FRT_flag',
    # PV plant 2 (PV2) signals.
    'DEV5/P': 'PV2/P',
    'DEV5/Q': 'PV2/Q',
    'DEV5/V0': 'PV2/V0',
    'DEV5/V1': 'PV2/V1',
    'DEV5/V2': 'PV2/V2',
    'DEV5/I0': 'PV2/I0',
    'DEV5/I1': 'PV2/I1',
    'DEV5/I2': 'PV2/I2',
    'WECC_PVPark_2/Converter_control/Control/GridControl_DLL/FRT_flag': 'PV2/FRT_flag',
}
name_pairs_four = {
    # PV plant 3 (PV3) signals.
    'DEV6/P': 'PV3/P',
    'DEV6/Q': 'PV3/Q',
    'DEV6/V0': 'PV3/V0',
    'DEV6/V1': 'PV3/V1',
    'DEV6/V2': 'PV3/V2',
    'DEV6/I0': 'PV3/I0',
    'DEV6/I1': 'PV3/I1',
    'DEV6/I2': 'PV3/I2',
    'WECC_PVPark_3/Converter_control/Control/GridControl_DLL/FRT_flag': 'PV3/FRT_flag',
}

In [ ]:
# Dictionary holding DataFrames of signals from all simulations.
data = {}
data['README'] = read_me
for key, index in zip(sim_keys, range(nsim)):
    # Export signals to DataFrame.
    sim = sims.get_simulation(index)
    sim_df = sim.to_dataframe()

    # Rename columns for readability.
    if variant in ['V1', 'V2', 'V3', 'V4']:
        sim_df.rename(columns=name_pairs, inplace=True)
    if variant in ['V2', 'V3', 'V4']:
        sim_df.rename(columns=name_pairs_two, inplace=True)
    if variant in ['V4']:
        sim_df.rename(columns=name_pairs_four, inplace=True)

    # Assign DataFrame to a simulation key.
    data[key] = sim_df

In [ ]:
# Pickle data to the external file.
file_name = variant + '-' + sc_time + '.pkl'
with open(file=file_name, mode='wb') as fp:
    pickle.dump(data, fp)